In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "cat_ejec_dbricks")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_consolidado = spark.table(f"{catalogo}.{esquema_source}.suspen_reinsta")


In [0]:
df_consolidado = df_consolidado.dropna(subset=["PERIODO", "CICLO", "ESTRATO"])

In [0]:
df_final = df_consolidado\
    .groupBy(col("ESTRATO"), col("CICLO"), col("PERIODO")).agg(
                max(col("DEUDA")).alias("DEUDA_MAXIMA"),
                avg(col("DEUDA")).alias("DEUDA_PROMEDIO"),                    
                avg(col("DIAS_EJEC_SUSP")).cast("long").alias("AVG_DIAS_EJEC_SUSP"),
                avg(col("DIAS_EJEC_REINS")).cast("long").alias("AVG_DIAS_EJEC_REINS"),
                min(col("DIAS_REINSTALACION")).cast("long").alias("REINSTALACION_MINIMA"),
                max(col("DIAS_REINSTALACION")).cast("long").alias("REINSTALACION_MAXIMA"),
                max(col("DIAS_MORA")).cast("long").alias("MORA_MAXIMA"),
                sum(col("VECES")).alias("SUSPENSIONES")
        ).orderBy(col("ESTRATO"), col("CICLO"), col("PERIODO"))

In [0]:
#df_final.describe()
df_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.suspen_reinsta_resultados")

In [0]:
%sql
select  *
from    cat_ejec_dbricks.golden.suspen_reinsta_resultados